# Phase 1 Offset Learning Usage

This notebook demonstrates the Phase 1 offset-learning workflow for a single crystal pair.

The idea is:

1. run the same rocking scan on the simulator and on the hardware backend
2. fit the center of each rocking curve
3. compute the offset as `hardware_center - simulator_center`
4. optionally write that offset to the matching EPICS offset PV

This notebook is set up to run end to end in simulation first, using a second simulator instance as a stand-in for the hardware backend. Once the workflow looks right, the same notebook can be pointed at the real hardware backend.

In [ ]:
from bluesky import RunEngine
from IPython.display import display

from lcls_beamline_toolbox.models.split_and_delay_motion import SND
from lcls_beamline_toolbox.models.split_and_delay_ophyd import SndOphyd
from lcls_beamline_toolbox.utility.phase1_offset_calibration import (
    learn_phase1_offset,
    write_phase1_offset,
)

import matplotlib.pyplot as plt
import pandas as pd

MOTOR_ATTR = "t1_th1"
DETECTOR_ATTR = "t1_dh_sum"
OFFSET_PV = "XCS:USER:SND:T1_TH1_OFFSET"

START = -100e-6
STOP = 100e-6
STEPS = 81
SHOTS_PER_STEP = 1

## Demo Setup

For a self-contained demo, use one simulator as the model-side backend and a second simulator as a stand-in for the hardware backend.

The stand-in "hardware" backend uses `set_current_position(...)` to shift the reported motor zero without changing the underlying optical state. That is a better stand-in for offset learning than physically misaligning the crystal.

In [ ]:
snd_sim = SND(9500)
snd_hw_demo = SND(9500)

known_demo_offset = 25e-6
getattr(snd_hw_demo, MOTOR_ATTR).set_current_position(known_demo_offset)

sim_backend = SndOphyd(snd_sim)
hw_backend = SndOphyd(snd_hw_demo)

RE_sim = RunEngine({})
RE_hw = RunEngine({})

result = learn_phase1_offset(
    RE_sim=RE_sim,
    sim_backend=sim_backend,
    RE_hw=RE_hw,
    hw_backend=hw_backend,
    motor_attr=MOTOR_ATTR,
    detector_attr=DETECTOR_ATTR,
    offset_pv=OFFSET_PV,
    start=START,
    stop=STOP,
    steps=STEPS,
    shots_per_step=SHOTS_PER_STEP,
    move=True,
    write=False,
)

summary = pd.DataFrame(
    [
        {
            "motor": result["motor"],
            "detector": result["detector"],
            "offset_pv": result["offset_pv"],
            "sim_center_urad": result["sim_center"] * 1e6,
            "hw_center_urad": result["hw_center"] * 1e6,
            "learned_offset_urad": result["offset"] * 1e6,
            "learned_offset_deg": result["offset_deg"],
            "known_demo_offset_urad": known_demo_offset * 1e6,
        }
    ]
)

display(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, curve, title in [
    (axes[0], result["sim_result"], "Simulator scan"),
    (axes[1], result["hw_result"], "Hardware-side scan (demo backend)"),
]:
    ax.plot(curve["x"] * 1e6, curve["y"], ".-")
    ax.axvline(curve["center"] * 1e6, color="k", linestyle=":", label="fit center")
    ax.set_title(title)
    ax.set_xlabel("motor position (urad)")
    ax.set_ylabel("diagnostic signal")
    ax.grid(True)

axes[0].legend(loc="best")
plt.tight_layout()
plt.show()

## Using the Real Hardware Backend

To use the real instrument instead of the demo hardware simulator:

1. replace `hw_backend = SndOphyd(snd_hw_demo)` with the real hardware backend object
2. replace `RE_hw = RunEngine({})` with the RunEngine you want to use for hardware scans
3. keep `motor_attr`, `detector_attr`, and `offset_pv` matched to the crystal you are calibrating
4. first run with `write=False` and inspect the fitted centers and learned offset
5. only after confirming the sign convention, write the PV

In [ ]:
# Example of the real-hardware call shape
#
# result = learn_phase1_offset(
#     RE_sim=RE_sim,
#     sim_backend=sim_backend,
#     RE_hw=RE_hw,
#     hw_backend=hw_backend,
#     motor_attr="t1_th1",
#     detector_attr="t1_dh_sum",
#     offset_pv="XCS:USER:SND:T1_TH1_OFFSET",
#     start=-100e-6,
#     stop=100e-6,
#     steps=81,
#     shots_per_step=1,
#     move=True,
#     write=False,
# )
#
# # After checking the fitted centers and sign convention:
# # write_phase1_offset(result)